In [1]:
# ============================================================================
# STEP 1: INSTALL REQUIRED PACKAGESss
# ============================================================================
print("="*70)
print(" INSTALLING PACKAGES")
print("="*70)
print("This will take 2-3 minutes. Please wait...")
print("(Dependency warnings are normal in Colab and can be ignored)\n")

!pip install -q langchain langchain-community langchain-groq sentence-transformers chromadb groq pandas 2>&1 | grep -v "dependency conflicts\|incompatible\|ERROR: pip's dependency" || true

print("\n Installation complete!")
print("="*70)

 INSTALLING PACKAGES
This will take 2-3 minutes. Please wait...
(Dependency warnings are normal in Colab and can be ignored)


 Installation complete!


In [2]:
!pip install -q google-search-results
print("\n done")


 done


In [3]:
# ============================================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================================
print("\n Importing libraries...")

import getpass
import os
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# ✅ Updated LangChain imports (modern)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.chat_history import InMemoryChatMessageHistory


# Prompts & schema
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document

# Optional (only if using web search / wikipedia)
from langchain_community.utilities import SerpAPIWrapper
from langchain_community.retrievers import WikipediaRetriever

# LCEL (modern chains)
from langchain_core.runnables import Runnable, RunnableLambda, RunnableMap

print(" All libraries imported successfully!")


 Importing libraries...
 All libraries imported successfully!


In [4]:
# ===================== STEP 3: LOAD API KEYS FROM .env =====================

import os
from dotenv import load_dotenv

print("\nLoading API keys from .env...")

# Load .env file
load_dotenv()

# ------------------ GROQ ------------------
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("Groq API key loaded ✅")

# ------------------ SERP API ------------------
# Get key from Colab secrets
serpapi_key = userdata.get("SERPAPI_API_KEY")

# Set environment variable
os.environ["SERPAPI_API_KEY"] = serpapi_key if serpapi_key else ""

if serpapi_key:
    web_search_enabled = True
    print("SerpAPI key loaded ✅ (Web search enabled)")
else:
    web_search_enabled = False
    print("No SerpAPI key found ⚠️ (Web search disabled)")

print("\nAll API keys configured!")
print("=" * 70)


Loading API keys from .env...
Groq API key loaded ✅
SerpAPI key loaded ✅ (Web search enabled)

All API keys configured!


In [5]:
# ============================================================================
# STEP 4: DEFINE PIPELINES (CLEAN + URL BASED + PDF READY)
# ============================================================================
print("\n" + "="*70)
print(" DEFINING PIPELINES")
print("="*70)

import os
from langchain_groq import ChatGroq
from langchain_community.document_loaders import TextLoader, PyPDFLoader, WebBaseLoader

# ------------------ LLM ------------------
llm = ChatGroq(
    model="llama-3-8b-instant",
    temperature=0.0,
    max_tokens=1024
)

# ------------------ 1. LLM ONLY ------------------
def llm_only_pipeline(question):
    return llm.invoke(question).content


# ------------------ 2. DOCUMENT RAG (LOCAL FILE) ------------------
print("\n Checking for local policy file...")
from google.colab import files
uploaded = files.upload()

doc_path = list(uploaded.keys())[0]   # auto get filename
print(f"Using file: {doc_path}")

doc_retriever = None

if os.path.exists(doc_path):
    print(" Found policies.txt")

    loader = TextLoader(doc_path)
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    splits = text_splitter.split_documents(docs)

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vectorstore = Chroma.from_documents(splits, embeddings)
    doc_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

else:
    print(" policies.txt not found")

def doc_rag_pipeline(question):
    if not doc_retriever:
        return "[Document RAG not available: add policies.pdf or policies.txt]"

    docs = doc_retriever.invoke(question)
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
Use ONLY the context below to answer the question.
If answer is not present, say you don't know.

Context:
{context}

Question: {question}
Answer:
"""
    return llm.invoke(prompt).content


# ------------------ 3. WIKIPEDIA (URL BASED) ------------------
print("\n Preparing Wikipedia URL pipeline...")

wiki_urls = [
    "https://en.wikipedia.org/wiki/Anthropic",
    "https://en.wikipedia.org/wiki/Claude_(language_model)"
]

wiki_docs = []
for url in wiki_urls:
    loader = WebBaseLoader(url)
    wiki_docs.extend(loader.load())

if wiki_docs:
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    wiki_splits = splitter.split_documents(wiki_docs)

    wiki_vectorstore = Chroma.from_documents(
        wiki_splits,
        HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    )

    wiki_retriever = wiki_vectorstore.as_retriever(search_kwargs={"k": 3})
else:
    wiki_retriever = None

def wikipedia_rag_pipeline(question):
    if not wiki_retriever:
        return "[Wikipedia data not loaded]"

    docs = wiki_retriever.invoke(question)
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
Answer using the Wikipedia context below.

Context:
{context}

Question: {question}
Answer:
"""
    return llm.invoke(prompt).content
print ("\n WIKIPEDIA DONE\n")

# ------------------ 4. SERP API ------------------
from langchain_community.utilities import SerpAPIWrapper

if web_search_enabled:
    serpapi = SerpAPIWrapper()

    def serpapi_rag_pipeline(question):
        context = serpapi.run(question)

        prompt = f"""
Answer using the real-time search results below.

Context:
{context}

Question: {question}
Answer:
"""
        return llm.invoke(prompt).content

else:
    def serpapi_rag_pipeline(question):
        return "[SerpAPI not enabled: Add SERPAPI_API_KEY to .env]"
print ("\nSERP API DONE\n")



 DEFINING PIPELINES

 Checking for local policy file...


Saving Policy.txt to Policy (4).txt
Using file: Policy (4).txt
 Found policies.txt


/tmp/ipykernel_55693/1596388085.py:46: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Preparing Wikipedia URL pipeline...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 WIKIPEDIA DONE


SERP API DONE



In [6]:
# ============================================================================
# STEP 5: PROCESS AND INDEX SOURCES (SEPARATE VECTOR STORES)
# ============================================================================
print("\n" + "="*70)
print(" PROCESSING MULTI-SOURCE DATA")
print("="*70)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# ------------------ COMMON COMPONENTS ------------------
print("\n Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print(" Embedding model loaded")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# ------------------ 1. DOCUMENT (TXT) ------------------
doc_retriever = None

if 'docs' in globals() and docs:
    print("\n Processing policies.txt...")

    splits = text_splitter.split_documents(docs)
    print(f"    Created {len(splits)} chunks")

    vectorstore_doc = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        collection_name="doc_rag"
    )

    doc_retriever = vectorstore_doc.as_retriever(search_kwargs={"k": 4})
    print("    Document retriever ready")

else:
    print("\n policies.txt not loaded → skipping document RAG")


# ------------------ 2. WIKIPEDIA (URL DATA) ------------------
wiki_retriever = None

if 'wiki_docs' in globals() and wiki_docs:
    print("\n Processing Wikipedia data...")

    wiki_splits = text_splitter.split_documents(wiki_docs)
    print(f"    Created {len(wiki_splits)} chunks")

    vectorstore_wiki = Chroma.from_documents(
        documents=wiki_splits,
        embedding=embeddings,
        collection_name="wiki_rag"
    )

    wiki_retriever = vectorstore_wiki.as_retriever(search_kwargs={"k": 3})
    print("    Wikipedia retriever ready")

else:
    print("\n Wikipedia data not loaded → skipping")


print("\n All available sources indexed!")
print("="*70)


 PROCESSING MULTI-SOURCE DATA

 Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Embedding model loaded

 Processing policies.txt...
    Created 32 chunks
    Document retriever ready

 Processing Wikipedia data...
    Created 271 chunks
    Wikipedia retriever ready

 All available sources indexed!


In [20]:
# ============================================================================
# STEP 6: INITIALIZE LLM (CLEAN VERSION)
# ============================================================================
print("\n" + "="*70)
print(" INITIALIZING AI COMPONENTS")
print("="*70)

from langchain_groq import ChatGroq

print("\n Initializing Groq LLM...")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,              # deterministic (better for RAG)
    max_tokens=1024,
)

print("    LLM ready")

# ------------------ SERP API ------------------
from langchain_community.utilities import SerpAPIWrapper

if web_search_enabled:
    print("\n Setting up web search...")

    serpapi = SerpAPIWrapper()

    def web_search(query):
        try:
            return serpapi.run(query)
        except Exception as e:
            return f"Web search error: {str(e)}"

    print("    Web search ready")

else:
    print("\n Web search disabled (no SERPAPI key)")
    serpapi = None
    web_search = None

print("\n All AI components ready!")
print("="*70)


 INITIALIZING AI COMPONENTS

 Initializing Groq LLM...
    LLM ready

 Setting up web search...
    Web search ready

 All AI components ready!


In [10]:
# ============================================================================
# STEP 7: QUERY ROUTER (DECIDES WHICH SOURCE TO USE)
# ============================================================================

def route_query(question):
    q = question.lower()

    # ------------------ DOCUMENT (XYZ POLICIES) ------------------
    if "xyz" in q :
        return "doc"

    # ------------------ RECENT / NEWS ------------------
    if "latest" in q or "recent" in q or "news" in q or "launch" in q:
        return "serp"

    # ------------------ WIKIPEDIA ------------------
    if "anthropic" in q or "claude" in q or "company" in q:
        return "wiki"

    # ------------------ DEFAULT ------------------
    return "llm"


# Smart pipeline caller
def smart_qa(question):
    source = route_query(question)

    print(f"\n[Using: {source.upper()}]")

    if source == "doc":
        return doc_rag_pipeline(question)

    elif source == "wiki":
        return wikipedia_rag_pipeline(question)

    elif source == "serp":
        return serpapi_rag_pipeline(question)

    else:
        return llm_only_pipeline(question)

In [ ]:
# ============================================================================
# STEP 10: INTERACTIVE MODE (SMART ROUTING)
# ============================================================================

print("\n" + "="*70)
print(" INTERACTIVE MODE (SMART RAG)")
print("="*70)

print("\nAsk questions!")
print("Type 'quit' to exit.\n")

while True:
    try:
        user_input = input("\n You: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ['quit', 'exit', 'q']:
            print("\n Exiting...")
            break

        print("\n" + "="*60)
        print(f"QUESTION: {user_input}")
        print("="*60)

        answer = smart_qa(user_input)

        print("\n Answer:\n")
        print(answer)

        print("\n" + "="*60)

    except Exception as e:
        print(f"\n Error: {str(e)}")


 INTERACTIVE MODE (SMART RAG)

Ask questions!
Type 'quit' to exit.


 You: notice period for regular companies

QUESTION: notice period for regular companies

[Using: LLM]

 Answer:

The notice period for regular companies can vary depending on the country, state, or industry. However, here are some general guidelines:

**In the United States:**

- For most employees, the notice period is typically 2 weeks (14 days) as per the federal law.
- Some states have their own laws regarding notice periods, such as:
  - California: 72 hours (3 days) for most employees, but 30 days for certain employees, such as those with a fixed-term contract.
  - New York: 2 weeks (14 days) for most employees, but 30 days for certain employees, such as those with a fixed-term contract.
  - Massachusetts: 2 weeks (14 days) for most employees, but 60 days for certain employees, such as those with a fixed-term contract.

**In the United Kingdom:**

- The notice period is typically 1 week (5 days) for employees 